# OFFLINE EVALUATION
## Integration using Langfuse Dashboard

#### Load the libraries
- LangFuse Client SDK
- RAGAS
- dotEnv (for Secrets)
- Pandas
- Postgres Client SDK (PsycoPG)

In [14]:
import os
import json
from datetime import datetime, timedelta, timezone

import pandas as pd
import psycopg2
from psycopg2.extras import RealDictCursor, DictCursor

from dotenv import load_dotenv

import langfuse
import ragas

print("Langfuse version:", getattr(langfuse, "__version__", "unknown"))
print("RAGAS version:", ragas.__version__)

Langfuse version: 4.14.2
RAGAS version: 0.2.15


#### Load the secrets through .env local file

In [2]:
load_dotenv()

LOOKBACK_MINUTES = 600

LANGFUSE_PUBLIC_KEY = os.getenv("LANGFUSE_PUBLIC_KEY")
LANGFUSE_SECRET_KEY = os.getenv("LANGFUSE_SECRET_KEY")
LANGFUSE_HOST = os.getenv("LANGFUSE_HOST", "http://localhost:3000")

POSTGRES_HOST = os.getenv("POSTGRES_HOST", "localhost")
POSTGRES_PORT = int(os.getenv("POSTGRES_PORT", "5432"))
POSTGRES_DB = os.getenv("POSTGRES_DB", 'week4')
POSTGRES_USER = os.getenv("POSTGRES_USER", 'ai_user')
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD", 'ai_password')

required_vars = {
    "LANGFUSE_PUBLIC_KEY": LANGFUSE_PUBLIC_KEY,
    "LANGFUSE_SECRET_KEY": LANGFUSE_SECRET_KEY,
    "POSTGRES_DB": POSTGRES_DB,
    "POSTGRES_USER": POSTGRES_USER,
    "POSTGRES_PASSWORD": POSTGRES_PASSWORD,
}

missing = [key for key, value in required_vars.items() if not value]

if missing:
    raise ValueError(f"Missing environment variables: {missing}")

print("Configuration loaded successfully.")
print("Langfuse host:", LANGFUSE_HOST)
print(
    f"PostgreSQL: {POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"
)

Configuration loaded successfully.
Langfuse host: http://localhost:3000
PostgreSQL: localhost:5432/week4


## PostGres - Fetch Workflow recorded data

In [10]:
def get_postgres_connection():
    return psycopg2.connect(
        host=POSTGRES_HOST,
        port=POSTGRES_PORT,
        dbname=POSTGRES_DB,
        user=POSTGRES_USER,
        password=POSTGRES_PASSWORD,
    )


conn = get_postgres_connection()

with conn.cursor() as cur:
    cur.execute("SELECT NOW();")
    print("PostgreSQL connection successful:", cur.fetchone()[0])

PostgreSQL connection successful: 2026-08-23 07:50:48.704254+00:00


In [4]:
end_time = datetime.now(timezone.utc)
start_time = end_time - timedelta(minutes=LOOKBACK_MINUTES)

print("Evaluation window:")
print("From:", start_time.isoformat())
print("To:  ", end_time.isoformat())

Evaluation window:
From: 2026-08-22T21:21:50.820247+00:00
To:   2026-08-23T07:21:50.820247+00:00


#### DB (postgres) extraction helper functions

In [5]:
def safe_json_loads(value):
    """
    Parse JSON safely.

    Returns:
        dict/list if parsing succeeds
        original value if it is already a Python object
        {} for None or invalid JSON
    """
    if value is None:
        return {}

    if isinstance(value, (dict, list)):
        return value

    if isinstance(value, str):
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            return {}

    return {}


def extract_request_metadata(raw_metadata):
    """
    Extract LiteLLM requester_metadata from the nested
    Langfuse / OTEL metadata structure.
    """

    level_1 = safe_json_loads(raw_metadata)

    attributes = level_1.get("attributes", {})

    nested_metadata = attributes.get("metadata", {})

    level_2 = safe_json_loads(nested_metadata)

    requester_metadata = level_2.get(
        "requester_metadata",
        {}
    )

    return {
        "request_id": requester_metadata.get("request_id"),
        "session_id": requester_metadata.get("session_id"),
        "route": requester_metadata.get("route"),
        "request_time": requester_metadata.get("request_time"),
    }

**OPTIONAL: Test the helper functions**

In [6]:
sample_metadata = {
    "attributes": {
        "metadata": json.dumps({
            "requester_metadata": {
                "request_id": "2116",
                "session_id": "ABC123",
                "route": "hr"
            }
        })
    }
}

extract_request_metadata(sample_metadata)

{'request_id': '2116',
 'session_id': 'ABC123',
 'route': 'hr',
 'request_time': None}

**Helper functions - Fetch PostGres Records
by \<SessionId and RequestId\>**

In [22]:
def list_inference_record_ids(conn):
    '''

    :param conn:
    :return:
    '''
    sql = """
        SELECT
            session_id,
            request_id
        FROM public.day25_inference_logs
        ORDER BY request_id DESC;
    """

    with conn.cursor(
        cursor_factory=DictCursor
    ) as cur:
        cur.execute(
            sql
        )
        return cur.fetchall()


def get_inference_record(
    conn,
    session_id,
    request_id
):
    '''
    Retrieves a single record given the Primary Key: (session_id, request_id)
    :param conn:
    :param session_id:
    :param request_id:
    :return:
    '''
    sql = """
        SELECT
            session_id,
            request_id,
            created_at,
            question,
            answer,
            context,
            rag_used,
            query_type,
            langfuse_trace_id,
            error,
            groundedness,
            relevancy,
            context_precision,
            context_recall,
            evaluated_at
        FROM public.day25_inference_logs
        WHERE session_id = %s
          AND request_id = %s
    """

    with conn.cursor(
        cursor_factory=RealDictCursor
    ) as cur:

        cur.execute(
            sql,
            (session_id, request_id)
        )

        return cur.fetchone()

##### Filter records excluding:
   - Router-only generations.
   - non-RAG requests.
   - failed requests.
   - records already evaluated.

In [20]:

def is_evaluation_candidate(record):
    '''
    Filters records avoiding:

    - Router-only generations.
    - non-RAG requests.
    - failed requests.
    - records already evaluated.
    :param record:
    :return:
    '''
    if record is None:
        return False

    if not record["rag_used"]:
        return False

    if record["error"] is not None:
        return False

    if record["evaluated_at"] is not None:
        return False

    if not record["question"]:
        return False

    if not record["answer"]:
        return False

    if not record["context"]:
        return False

    return True

##### Test record fetch from DB

In [33]:
ll = list_inference_record_ids(conn)
print(ll)

[['ABC123', '2155'], ['ABC123', '2147'], ['ABC123', '2142']]


In [12]:
test_record = get_inference_record(
    conn,
    session_id="ABC123",
    request_id="2142"
)

if test_record:
    print("Record found.")
    print("Question:", test_record["question"])
    print("RAG used:", test_record["rag_used"])
    print("Query type:", test_record["query_type"])
    print("Already evaluated:", test_record["evaluated_at"] is not None)

    print("\nNumber of contexts:", len(test_record["context"]))

    for i, ctx in enumerate(test_record["context"], start=1):
        print(f"\n--- Context {i} ---")
        print(ctx[:300])
else:
    print("No record found.")

Record found.
Question: As a new joiniee how many leaves am I entitled to, in a year, as per the company policy?
RAG used: True
Query type: hr
Already evaluated: False

Number of contexts: 3

--- Context 1 ---
tems remotely.
4 Time Off and Leave Policy
Work-life balance is essential. Full-Time employees are entitled to the following annual leave
allocations:
• Annual Leave: 18 days. Requests should normally be submitted at least two weeks in
advance via the HR portal.
• Medical Leave: 10 days. A medical c

--- Context 2 ---
tems remotely.
4 Time Off and Leave Policy
Work-life balance is essential. Full-Time employees are entitled to the following annual leave
allocations:
• Annual Leave: 18 days. Requests should normally be submitted at least two weeks in
advance via the HR portal.
• Medical Leave: 10 days. A medical c

--- Context 3 ---
Employees may be classified into the following categories: Full-Time, Contract, Intern, or Con-
sultant. Benefits and eligibility criteria vary by emp

## Langfuse Traces Fetch
``` BATCH_SIZE = 100 ```

In [40]:
BATCH_SIZE = 100

In [24]:
# Initialize Langfuse client with an explicit URL
from langfuse import Langfuse

langfuse_client = Langfuse(
    public_key=LANGFUSE_PUBLIC_KEY,
    secret_key=LANGFUSE_SECRET_KEY,
    host=LANGFUSE_HOST,
)

print("Langfuse client initialized.")

Langfuse client initialized.


In [41]:
recent_traces_response = langfuse_client.api.trace.list(
    limit=BATCH_SIZE
)

#print(type(recent_traces_response))
print(f"Number of LangFuse Records Fetched: "
      f"{len(recent_traces_response.data)}")

#print(recent_traces_response.data[0])

Number of LangFuse Records Fetched: 12


#### LanfFuse Data Wrangling Helper Methods

In [38]:
def to_dict(obj):
    """
    Convert Langfuse/Pydantic objects or dictionaries
    into a standard Python dictionary.
    """

    if obj is None:
        return {}

    if isinstance(obj, dict):
        return obj

    if hasattr(obj, "model_dump"):
        return obj.model_dump()

    if hasattr(obj, "dict"):
        return obj.dict()

    if hasattr(obj, "__dict__"):
        return obj.__dict__

    return {}

In [39]:
response_dict = to_dict(recent_traces_response)

trace_items = response_dict.get(
    "data",
    []
)

print("Total traces returned:", len(trace_items))


def parse_timestamp(value):
    if value is None:
        return None

    if isinstance(value, datetime):
        return value

    if isinstance(value, str):
        return datetime.fromisoformat(
            value.replace("Z", "+00:00")
        )

    return None


recent_traces = []

for trace in trace_items:

    trace_dict = to_dict(trace)

    timestamp = parse_timestamp(
        trace_dict.get("timestamp")
        or trace_dict.get("createdAt")
        or trace_dict.get("created_at")
    )

    if timestamp is None:
        continue

    if timestamp.tzinfo is None:
        timestamp = timestamp.replace(
            tzinfo=timezone.utc
        )

    if start_time <= timestamp <= end_time:
        recent_traces.append(trace_dict)


print(
    "Traces inside evaluation window:",
    len(recent_traces)
)

Total traces returned: 12
Traces inside evaluation window: 4


#### Print Raw Traces

In [42]:
for i, trace in enumerate(
    # recent_traces[:10],
    trace_items[:BATCH_SIZE],
    start=1
):
    print(f"\n{'=' * 80}")
    print("TRACE", i)
    print("ID:", trace.get("id"))
    print("Name:", trace.get("name"))
    print(
        "Timestamp:",
        trace.get("timestamp")
        or trace.get("createdAt")
    )
    print("Session:", trace.get("sessionId"))
    print("Metadata:", trace.get("metadata"))


TRACE 1
ID: f1740d03d5c8ab2465d97e7ee82c626f
Name: Execute evaluator: Custom_EVAL
Timestamp: 2026-08-23 08:17:05.159000+00:00
Session: None
Metadata: {'job_execution_id': '2b224acc8f1dfe28a6f8c7f44933594a', 'job_configuration_id': 'ed0199e8-f505-4480-a64c-e7a2fa9ee9cb', 'target_trace_id': 'a348a3f2650232214185d3ebc3966b26', 'target_observation_id': '67bdb1cf45711dea'}

TRACE 2
ID: a348a3f2650232214185d3ebc3966b26
Name: litellm_request
Timestamp: 2026-08-23 08:16:58.922000+00:00
Session: None
Metadata: {'attributes': {'metadata': '{"user_api_key_hash": "DUMMY", "user_api_key_alias": null, "user_api_key_spend": 0.0, "user_api_key_max_budget": null, "user_api_key_budget_reset_at": null, "user_api_key_team_id": null, "user_api_key_org_id": null, "user_api_key_project_id": null, "user_api_key_user_id": null, "user_api_key_team_alias": null, "user_api_key_user_email": null, "user_api_key_end_user_id": null, "user_api_key_request_route": "/responses", "spend_logs_metadata": null, "requester_

#### Build normalized Langfuse candidates

This cell first attempts to obtain the metadata directly from the trace.

In [43]:
langfuse_candidates = []

# for trace in recent_traces:
for trace in trace_items:
    trace_id = trace.get("id")

    metadata = trace.get("metadata", {})

    request_metadata = extract_request_metadata(
        metadata
    )

    request_id = request_metadata.get(
        "request_id"
    )

    session_id = request_metadata.get(
        "session_id"
    )

    route = request_metadata.get(
        "route"
    )

    # Fallback to normal Langfuse fields
    if session_id is None:
        session_id = (
            trace.get("sessionId")
            or trace.get("session_id")
        )

    # Some metadata may already be a direct dict
    if request_id is None and isinstance(
        metadata,
        dict
    ):
        request_id = metadata.get(
            "request_id"
        )

    if request_id and session_id and route:

        langfuse_candidates.append(
            {
                "trace_id": trace_id,
                "session_id": session_id,
                "request_id": request_id,
                "route": route,
                "timestamp": (
                    trace.get("timestamp")
                    or trace.get("createdAt")
                ),
                "raw_trace": trace,
            }
        )


candidate_df = pd.DataFrame(
    langfuse_candidates
)

display(candidate_df)

,trace_id,session_id,request_id,route,timestamp,raw_trace
0,a348a3f2650232214185d3ebc3966b26,ABC123,2155,hr,2026-08-23 08:16:58.922000+00:00,"{'id': 'a348a3f2650232214185d3ebc3966b26', 'ti..."
1,5c09ffa14cd9fa33a05866f4ba4fe341,ABC123,2147,hr,2026-08-23 07:20:12.986000+00:00,"{'id': '5c09ffa14cd9fa33a05866f4ba4fe341', 'ti..."
2,ca03eb5085fd6e385f0843e2558d98c3,ABC123,2142,hr,2026-08-22 16:29:06.788000+00:00,"{'id': 'ca03eb5085fd6e385f0843e2558d98c3', 'ti..."


## JOIN: LangFuse-record * PostGres-Record
#### Match Langfuse candidates with PostgreSQL

* Use Postgres to filter out unwanted candidates (i.e. those which are already evaluated, or are not relevant etc for instance)

In [44]:
matched_candidates = []

for candidate in langfuse_candidates:

    record = get_inference_record(
        conn=conn,
        session_id=candidate["session_id"],
        request_id=candidate["request_id"],
    )

    if record is None:
        print(
            "No PostgreSQL record:",
            candidate["session_id"],
            candidate["request_id"]
        )
        continue

    candidate["postgres_record"] = record

    candidate["is_candidate"] = (
        is_evaluation_candidate(record)
    )

    matched_candidates.append(candidate)


print(
    "Langfuse/PostgreSQL matches:",
    len(matched_candidates)
)

evaluation_candidates = [
    item
    for item in matched_candidates
    if item["is_candidate"]
]

print(
    "Eligible for evaluation:",
    len(evaluation_candidates)
)

Langfuse/PostgreSQL matches: 3
Eligible for evaluation: 2


#### Inspect the records selected for evaluation

In [45]:
for item in evaluation_candidates:

    record = item["postgres_record"]

    print("\n" + "=" * 80)

    print("TRACE ID:", item["trace_id"])
    print("SESSION ID:", item["session_id"])
    print("REQUEST ID:", item["request_id"])
    print("ROUTE:", item["route"])

    print("\nQUESTION:")
    print(record["question"])

    print("\nANSWER:")
    print(record["answer"])

    print(
        "\nCONTEXT COUNT:",
        len(record["context"])
    )
    # print(record["context"][2])

display(pd.DataFrame(evaluation_candidates))



TRACE ID: a348a3f2650232214185d3ebc3966b26
SESSION ID: ABC123
REQUEST ID: 2155
ROUTE: hr

QUESTION:
Can I take my office laptop while during an international office travel? What do I need to take care?

ANSWER:
**Taking your office laptop on international travel**

- **Yes, you may take your office laptop** when you travel abroad. The policy states that laptops are used whenever you access corporate resources from outside the company’s offices.  

> “Laptops … used whenever accessing corporate resources from outside company offices.”  

- **What you need to take care of:**  
  - **Use the VPN** for any connection to corporate systems while you are off‑site. The policy requires that a VPN be used whenever corporate resources are accessed from outside the office.  

> “VPN must be used whenever accessing corporate resources from outside company offices.”  

- **Travel‑related approval:**  
  - International travel itself requires **Vice President approval**.  

> “International travel r

,trace_id,session_id,request_id,route,timestamp,raw_trace,postgres_record,is_candidate
0,a348a3f2650232214185d3ebc3966b26,ABC123,2155,hr,2026-08-23 08:16:58.922000+00:00,"{'id': 'a348a3f2650232214185d3ebc3966b26', 'ti...","{'session_id': 'ABC123', 'request_id': '2155',...",True
1,5c09ffa14cd9fa33a05866f4ba4fe341,ABC123,2147,hr,2026-08-23 07:20:12.986000+00:00,"{'id': '5c09ffa14cd9fa33a05866f4ba4fe341', 'ti...","{'session_id': 'ABC123', 'request_id': '2147',...",True


## RAGAS - Offline EVAL

#### Build the RAGAS dataset

In [46]:
from datasets import Dataset

def build_ragas_dataset(evaluation_candidates):
    rows = []

    for item in evaluation_candidates:

        record = item["postgres_record"]

        rows.append(
            {
                "user_input": record["question"],
                "retrieved_contexts": list(
                    record["context"]
                ),
                "response": record["answer"],
            }
        )
        # rows.append(
        #     {
        #         "question": record["question"],
        #         "contexts": list(
        #             record["context"]
        #         ),
        #         "answer": record["answer"],
        #     }
        # )

    return Dataset.from_list(rows)


ragas_dataset = build_ragas_dataset(
    evaluation_candidates
)

print(ragas_dataset)
display(pd.DataFrame(ragas_dataset))
# display(ragas_dataset['contexts'][0][2])

Dataset({
    features: ['user_input', 'retrieved_contexts', 'response'],
    num_rows: 2
})


,user_input,retrieved_contexts,response
0,Can I take my office laptop while during an in...,[used whenever accessing corporate resources f...,**Taking your office laptop on international t...
1,Whose approval is needed for international off...,[international business travel. This policy go...,**Approval for international business travel**...


#### Summon 'LLM-AS-A-JUDGE'

In [47]:
# LangChain OpenAI integrations
from ragas.llms import LangchainLLMWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Verify the key is loaded (don't print the actual key in output!)
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found. Please check your .env file.")


EVAL_MODEL = os.getenv(
    "RAGAS_EVAL_MODEL",
    "gpt-4o-mini"
)


eval_llm = ChatOpenAI(
    model=EVAL_MODEL,
    temperature=0,
)

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

print(f"✅ Judge LLM configured as: {eval_llm.model_name}")
print(f"✅ Embeddings configured as: {embeddings.model}")

✅ Judge LLM configured as: gpt-4o-mini
✅ Embeddings configured as: text-embedding-3-small


#### Begin Evaluation

In [48]:
# Updated Ragas metric imports
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    summarization_score,
    context_precision,
    context_recall,
)
from ragas import evaluate

from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
    ContextPrecision,
)

print(faithfulness)

Faithfulness(_required_columns={<MetricType.SINGLE_TURN: 'single_turn'>: {'retrieved_contexts', 'response', 'user_input'}}, name='faithfulness', llm=None, output_type=<MetricOutputType.CONTINUOUS: 'continuous'>, nli_statements_prompt=NLIStatementPrompt(instruction=Your task is to judge the faithfulness of a series of statements based on a given context. For each statement you must return verdict as 1 if the statement can be directly inferred based on the context or 0 if the statement can not be directly inferred based on the context., examples=[(NLIStatementInput(context='John is a student at XYZ University. He is pursuing a degree in Computer Science. He is enrolled in several courses this semester, including Data Structures, Algorithms, and Database Management. John is a diligent student and spends a significant amount of time studying and completing assignments. He often stays late in the library to work on his projects.', statements=['John is majoring in Biology.', 'John is taking 

In [49]:
display(pd.DataFrame(ragas_dataset))
# display(pd.DataFrame(ragas_dataset)[['contexts']][0:1].values[0][0][2])

,user_input,retrieved_contexts,response
0,Can I take my office laptop while during an in...,[used whenever accessing corporate resources f...,**Taking your office laptop on international t...
1,Whose approval is needed for international off...,[international business travel. This policy go...,**Approval for international business travel**...


#### Run RAGAS evaluation

In [51]:
# Execute Ragas using the cloud models
score = evaluate(
    dataset=ragas_dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        # context_precision,
        # context_recall,
        # summarization_score,
    ],
    llm=eval_llm,
    embeddings=embeddings,
    # Prevents the entire run from failing if one evaluation errors out
    raise_exceptions=False
)


# Output as a clean Pandas DataFrame for analysis
df_results = score.to_pandas()
display(pd.DataFrame(df_results))

Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

,user_input,retrieved_contexts,response,faithfulness,answer_relevancy
0,Can I take my office laptop while during an in...,[used whenever accessing corporate resources f...,**Taking your office laptop on international t...,0.500000,0.897052
1,Whose approval is needed for international off...,[international business travel. This policy go...,**Approval for international business travel**...,0.666667,0.000000


#### Collate the EVAL back into the candidate record

In [52]:
def safe_score(value):
    if value is None:
        return None

    try:
        return float(value)
    except (TypeError, ValueError):
        return None


evaluation_results = []

for item, (_, row) in zip(
    evaluation_candidates,
    df_results.iterrows()
):

    evaluation_results.append(
        {
            "trace_id": item["trace_id"],
            "session_id": item["session_id"],
            "request_id": item["request_id"],

            "groundedness": safe_score(
                row.get("faithfulness")
            ),

            "relevancy": safe_score(
                row.get("answer_relevancy")
                or row.get("response_relevancy")
            ),

            "context_precision": safe_score(
                row.get("context_precision")
            ),

            # Not evaluated because no reference
            "context_recall": None,
        }
    )


results_df = pd.DataFrame(
    evaluation_results
)

results_df

,trace_id,session_id,request_id,groundedness,relevancy,context_precision,context_recall
0,a348a3f2650232214185d3ebc3966b26,ABC123,2155,0.500000,0.897052,None,None
1,5c09ffa14cd9fa33a05866f4ba4fe341,ABC123,2147,0.666667,NaN,None,None


## Write-BACK!

#### Update the LangFuse event record

In [53]:
def send_scores_to_langfuse(
    langfuse_client,
    result
):
    trace_id = result["trace_id"]

    scores = {
        "groundedness": result["groundedness"],
        "relevancy": result["relevancy"],
        "context_precision": result["context_precision"],
    }

    for score_name, score_value in scores.items():

        if score_value is None:
            continue

        langfuse_client.create_score(
            trace_id=trace_id,
            name=score_name,
            value=float(score_value),
            data_type="NUMERIC",
        )

In [54]:
for result in evaluation_results:

    send_scores_to_langfuse(
        langfuse_client,
        result
    )

langfuse_client.flush()

print("Langfuse scores submitted.")

Langfuse scores submitted.


#### Update the DB record (Postgres)

PostgreSQL update helper

In [55]:
def update_postgres_evaluation(
    conn,
    result
):
    sql = """
        UPDATE public.day25_inference_logs
        SET
            langfuse_trace_id = %s,
            groundedness = %s,
            relevancy = %s,
            context_precision = %s,
            context_recall = %s,
            evaluated_at = NOW()
        WHERE session_id = %s
          AND request_id = %s
    """

    with conn.cursor() as cur:

        cur.execute(
            sql,
            (
                result["trace_id"],
                result["groundedness"],
                result["relevancy"],
                result["context_precision"],
                result["context_recall"],
                result["session_id"],
                result["request_id"],
            )
        )

Update all evaluated records

In [56]:
try:

    for result in evaluation_results:

        update_postgres_evaluation(
            conn,
            result
        )

    conn.commit()

    print(
        "PostgreSQL evaluation results committed."
    )

except Exception as e:

    conn.rollback()

    print(
        "PostgreSQL update failed."
    )

    raise e

PostgreSQL evaluation results committed.


## Final verification

In [57]:
verification_rows = []

for result in evaluation_results:

    updated_record = get_inference_record(
        conn,
        result["session_id"],
        result["request_id"],
    )

    verification_rows.append(
        {
            "session_id":
                updated_record["session_id"],

            "request_id":
                updated_record["request_id"],

            "langfuse_trace_id":
                updated_record[
                    "langfuse_trace_id"
                ],

            "groundedness":
                updated_record["groundedness"],

            "relevancy":
                updated_record["relevancy"],

            "context_precision":
                updated_record[
                    "context_precision"
                ],

            "context_recall":
                updated_record[
                    "context_recall"
                ],

            "evaluated_at":
                updated_record["evaluated_at"],
        }
    )


verification_df = pd.DataFrame(
    verification_rows
)

display(verification_df)

,session_id,request_id,langfuse_trace_id,groundedness,relevancy,context_precision,context_recall,evaluated_at
0,ABC123,2155,a348a3f2650232214185d3ebc3966b26,0.500000,0.897052,None,None,2026-08-23 07:50:48.704254+00:00
1,ABC123,2147,5c09ffa14cd9fa33a05866f4ba4fe341,0.666667,NaN,None,None,2026-08-23 07:50:48.704254+00:00


!! DONE !!